In [1]:
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import os
import numpy as np
import torch
import albumentations as A


class faceRecoDatasetPascal_coord(Dataset):
    def __init__(self, image_dir, label2_dir, transform=None):
        if(isinstance(image_dir, Path)):
            self.image_list = list(image_dir.glob('*.jpg'))
        else:
            self.image_list = list(Path(image_dir).glob('*.jpg'))
        if(isinstance(label2_dir, Path)):
            self.label_dir = label2_dir
        else:
            self.label_dir = Path(label2_dir)
        self.transform = transform
    def __len__(self):
        return len(self.image_list)
    def __getitem__(self, index):
        image_path = self.image_list[index]
        image_name = image_path.stem
        label_path = self.label_dir/f"{image_name}.txt"
        img = Image.open(image_path).convert('RGB')
        img_np_arr = np.array(img, dtype=np.uint8)
        if os.path.exists(label_path) and os.path.getsize(label_path)>0:
            label_np_arr = np.loadtxt(label_path, usecols=(2, 3, 4, 5), dtype=np.float32)
            if label_np_arr.ndim == 1:
                label_np_arr = np.expand_dims(label_np_arr, axis=0)
        else:
            label_np_arr = np.empty((0, 4), dtype=np.float32)
        
        if len(label_np_arr) >0:
            bboxes = label_np_arr.tolist()
            class_labels = [0]*len(bboxes)
        else:
            bboxes = []
            class_labels = []
        
        if self.transform:
            transformed = self.transform(image=img_np_arr, bboxes=bboxes, class_labels=class_labels)
            img_tensor = transformed['image']
            transformed_label = transformed['bboxes']
            transformed_classes = transformed['class_labels']
        else:
            img_tensor = torch.from_numpy(img_np_arr).permute(2, 0, 1)/255.0
            transformed_label = label_np_arr
            transformed_classes = class_labels
        
        if len(transformed_label) >0:
            box_tensor = torch.tensor(transformed_label, dtype=torch.float32)
            classes_tensor = torch.tensor(transformed_classes, dtype=torch.long)
        else:
            box_tensor = torch.empty((0, 4), dtype=torch.float32)
            classes_tensor = torch.tensor(transformed_classes, dtype=torch.long)
        
        return img_tensor, {"boxes": box_tensor, "labels": classes_tensor}

In [2]:
dataset = faceRecoDatasetPascal_coord(
    image_dir='../data/images/val',
    label2_dir='../data/labels2',
    transform=None
)

In [3]:
image, label = next(iter(dataset))
image, label['boxes']

(tensor([[[0.5176, 0.5176, 0.5451,  ..., 0.1961, 0.2039, 0.2275],
          [0.5569, 0.5529, 0.5529,  ..., 0.1922, 0.2000, 0.2078],
          [0.5569, 0.5725, 0.5451,  ..., 0.2588, 0.2588, 0.2745],
          ...,
          [0.0275, 0.0353, 0.0392,  ..., 0.0353, 0.0353, 0.0353],
          [0.0392, 0.0353, 0.0275,  ..., 0.0353, 0.0392, 0.0353],
          [0.0275, 0.0392, 0.0431,  ..., 0.0353, 0.0353, 0.0353]],
 
         [[0.3255, 0.3137, 0.3490,  ..., 0.1176, 0.1216, 0.1333],
          [0.3725, 0.3647, 0.3765,  ..., 0.1059, 0.1333, 0.1020],
          [0.3882, 0.3882, 0.3608,  ..., 0.1765, 0.1843, 0.1843],
          ...,
          [0.0039, 0.0039, 0.0000,  ..., 0.0000, 0.0000, 0.0039],
          [0.0000, 0.0000, 0.0118,  ..., 0.0000, 0.0078, 0.0039],
          [0.0039, 0.0000, 0.0039,  ..., 0.0039, 0.0039, 0.0000]],
 
         [[0.1569, 0.1294, 0.1412,  ..., 0.0196, 0.0157, 0.0314],
          [0.1922, 0.1725, 0.1608,  ..., 0.0118, 0.0314, 0.0196],
          [0.1882, 0.1765, 0.1490,  ...,

In [4]:
class faceRecoDatasetYolo_coord(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        """
        image_dir-All images must be inside this folder
        label_dir-All labels must be inside this folder
        transform-Transforms must be of type albumentation.Compose
        """
        if(isinstance(image_dir, Path)):
            self.image_list = list(image_dir.glob('*.jpg'))
        else:
            self.image_list = list(Path(image_dir).glob('*.jpg'))
        if(isinstance(label_dir, Path)):
            self.label_dir = label_dir
        else:
            self.label_dir = Path(label_dir)
        self.transform = transform
    def __len__(self):
        return len(self.image_list)
    def __getitem__(self, index):
        image_path = self.image_list[index]
        image_name = image_path.stem
        label_path = self.label_dir/f"{image_name}.txt"
        img = Image.open(image_path).convert('RGB')
        img_np_arr = np.array(img, dtype=np.uint8)
        if os.path.exists(label_path) and os.path.getsize(label_path)>0:
            label_np_arr = np.loadtxt(label_path, usecols=(1, 2, 3, 4), dtype=np.float32)
            if label_np_arr.ndim == 1:
                label_np_arr = np.expand_dims(label_np_arr, axis=0)
        else:
            label_np_arr = np.empty((0, 4), dtype=np.float32)
        
        if len(label_np_arr) >0:
            bboxes = label_np_arr.tolist()
            class_labels = [0]*len(bboxes)
        else:
            bboxes = []
            class_labels = []
        
        if self.transform:
            transformed = self.transform(image=img_np_arr, bboxes=bboxes, class_labels=class_labels)
            img_tensor = transformed['image']
            transformed_label = transformed['bboxes']
            transformed_classes = transformed['class_labels']
        else:
            img_tensor = torch.from_numpy(img_np_arr).permute(2, 0, 1)/255.0
            transformed_label = label_np_arr
            transformed_classes = class_labels
        
        if len(transformed_label) >0:
            box_tensor = torch.tensor(transformed_label, dtype=torch.float32)
            classes_tensor = torch.tensor(transformed_classes, dtype=torch.long)
        else:
            box_tensor = torch.empty((0, 4), dtype=torch.float32)
            classes_tensor = torch.tensor(transformed_classes, dtype=torch.long)
        
        return img_tensor, {"boxes": box_tensor, "labels": classes_tensor}

In [5]:
dataset = faceRecoDatasetYolo_coord(
    image_dir='../data/images/val',
    label_dir='../data/labels/val',
    transform=None
)

In [8]:
image, label = dataset[10]
image, label

(tensor([[[0.5098, 0.5098, 0.5098,  ..., 0.3843, 0.3843, 0.3843],
          [0.5098, 0.5098, 0.5098,  ..., 0.3882, 0.3843, 0.3843],
          [0.5098, 0.5098, 0.5098,  ..., 0.3882, 0.3882, 0.3843],
          ...,
          [0.2588, 0.2039, 0.2667,  ..., 0.5529, 0.5333, 0.5059],
          [0.3137, 0.2863, 0.3412,  ..., 0.5333, 0.5294, 0.4980],
          [0.2588, 0.2627, 0.2824,  ..., 0.5333, 0.5294, 0.5294]],
 
         [[0.7529, 0.7529, 0.7529,  ..., 0.6353, 0.6353, 0.6353],
          [0.7529, 0.7529, 0.7529,  ..., 0.6392, 0.6353, 0.6353],
          [0.7529, 0.7529, 0.7529,  ..., 0.6392, 0.6392, 0.6353],
          ...,
          [0.2980, 0.2510, 0.3255,  ..., 0.4039, 0.3843, 0.3569],
          [0.3529, 0.3333, 0.4000,  ..., 0.3843, 0.3725, 0.3412],
          [0.2980, 0.3098, 0.3412,  ..., 0.3843, 0.3725, 0.3608]],
 
         [[0.9451, 0.9451, 0.9451,  ..., 0.9176, 0.9176, 0.9176],
          [0.9451, 0.9451, 0.9451,  ..., 0.9216, 0.9176, 0.9176],
          [0.9451, 0.9451, 0.9451,  ...,

In [9]:
NUM_WORKERS = os.cpu_count() or 1
def create_dataloaders(
        train_dir:str|Path,
        val_dir:str|Path,
        labels2_dir:str|Path,
        train_batch_size:int | None,
        val_batch_size:int | None,
        in_yolo:bool = False,
        transform:A.Compose|None = None,
        num_workers:int = NUM_WORKERS,
        train_shuffle:bool = True,
        val_shuffle:bool = False
):
    if(in_yolo):
        train_data = faceRecoDatasetYolo_coord(
            image_dir=train_dir,
            label_dir=labels2_dir,
            transform=transform
        )
        val_data = faceRecoDatasetYolo_coord(
            image_dir=val_dir,
            label_dir=labels2_dir,
            transform=transform
        )
        train_loader = DataLoader(
            dataset=train_data,
            batch_size=train_batch_size,
            shuffle=train_shuffle,
            num_workers=num_workers,
            pin_memory=True
        )
        test_loader = DataLoader(
            dataset=val_data,
            batch_size=val_batch_size,
            shuffle=val_shuffle,
            num_workers=num_workers,
            pin_memory=True
        )
    else:
        train_data = faceRecoDatasetPascal_coord(
            image_dir=train_dir,
            label2_dir=labels2_dir,
            transform=transform
        )
        val_data = faceRecoDatasetPascal_coord(
            image_dir=val_dir,
            label2_dir=labels2_dir,
            transform=transform
        )
        train_loader = DataLoader(
            dataset=train_data,
            batch_size=train_batch_size,
            shuffle=train_shuffle,
            num_workers=num_workers,
            pin_memory=True
        )
        test_loader = DataLoader(
            dataset=val_data,
            batch_size=val_batch_size,
            shuffle=val_shuffle,
            num_workers=num_workers,
            pin_memory=True
        )
    return train_loader, test_loader